# 04.4 — Convolutional backbones

Beyond the recurrent encoders, the project ships three convolutional ones (Milestone CC.1): `Convolutional`, `Resnet`, and `Multi-Filter Convolutional`. They plug into the **same composite slot** as the GRU — swap one config string and the encoder changes, everything else stays. This notebook is about how 1-D convolution applies to windowed neural data, and how these three variants differ.

This notebook covers:

* 1-D convolution intuition: a learned filter sliding over time.
* The channels-second `(N, C, T)` convention and how the conv encoders reach it.
* The three variants — plain conv, residual, multi-filter — and what each adds.
* Why they're interchangeable with the RNN encoder despite very different internals.

**Prerequisites:** [04.2 simple encoder](04.2_building_a_simple_encoder.ipynb), [04.3 RNN blocks](04.3_rnn_building_blocks.ipynb), [02.2 axis conventions](../02_numpy_and_pytorch_basics/02.2_array_axis_conventions.ipynb).

## Section 1 — What MATLAB does

The MATLAB conv encoders stack `convolution1dLayer` / `convolution2dLayer` with normalization and activation, and the SLURM sweep selects them by name (`ModelName = 'Convolutional' | 'Resnet' | 'Multi-Filter Convolutional'`), often with gradient accumulation because conv batches are memory-heavy:

```matlab
convolution1dLayer(FilterSize, NumFilters, 'Stride', Stride, 'Padding', 'same')
instanceNormalizationLayer         % 'Instance' normalization
leakyReluLayer
% ResNet variant adds skip connections; Multi-Filter runs several filter sizes in parallel.
```

The Python port keeps the names and the roles; the mechanics below are the same convolution, expressed in PyTorch.

## Section 2 — The Python concepts you need

### 2.1 — 1-D convolution: a filter sliding over time

A conv filter is a small learned pattern; convolution slides it across the time axis and records how strongly each position matches. One filter → one feature map; many filters → many. The key hyperparameters:

In [ ]:
import torch
from torch import nn

# Conv1d expects (batch, channels, length) — channels SECOND (02.2 §2.3, §5).
conv = nn.Conv1d(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1)
x = torch.randn(2, 4, 20)          # (batch=2, channels=4, length=20)
y = conv(x)
print("in :", tuple(x.shape), "(batch, channels, length)")
print("out:", tuple(y.shape), "(batch, out_channels=8 filters, length)")
print(f"params: {sum(p.numel() for p in conv.parameters())}  "
      f"= 8 filters × (4 in-channels × 3 kernel) + 8 biases")

**`kernel_size`** = filter width (how many timesteps it sees at once); **`stride`** = step between positions (2 halves the length); **`padding`** = zeros added at the edges so the output length is controllable (`padding=1` with `kernel_size=3` keeps length unchanged). Note the channels-second layout — the recurring trap from [02.2 §5](../02_numpy_and_pytorch_basics/02.2_array_axis_conventions.ipynb): a `(batch, length, channels)` tensor into `Conv1d` convolves the wrong axis.

### 2.2 — How the conv encoders fit the composite

Here's the elegant part: the composite hands EVERY encoder the same flat `(B, W, T*A*C)` tensor ([02.2 §2.3](../02_numpy_and_pytorch_basics/02.2_array_axis_conventions.ipynb)). An RNN consumes that directly. A conv encoder instead **reshapes it back to 5-D internally** — `(B, W, T, A, C)` — runs its kernels over the `(T, A)` plane, and flattens the result back so its output matches the RNN encoder's `(B, W, hidden)` contract. The interface is identical; only the middle differs.

In [ ]:
import neural_data_decoding.models
from neural_data_decoding.models.registry import build_encoder

T, A, C = 10, 2, 4                 # per-window dims; flat feature = T*A*C = 80
cfg = {"in_features": T*A*C, "samples_per_window": T, "num_areas": A,
       "hidden_sizes": [8, 16, 4], "dropout": 0.0,
       "want_normalization": "Instance", "activation": "Leaky ReLU", "stride": 2}

conv_enc = build_encoder("Convolutional", cfg)
x = torch.randn(2, 5, T*A*C)       # (B, W, flat) — SAME shape an RNN would get
print("input :", tuple(x.shape), "(batch, windows, flat features) — the composite's flat output")
print("output:", tuple(conv_enc(x).shape), "(batch, windows, hidden) — the SAME contract a GRU produces")

Input and output shapes match what an RNN encoder produces — which is exactly why the composite can swap them freely. The reshape-to-5-D-and-back is entirely inside the conv encoder (`ConvolutionalEncoder.forward`), invisible to the composite.

### 2.3 — The three variants

In [ ]:
for name in ["Convolutional", "Resnet", "Multi-Filter Convolutional"]:
    enc = build_encoder(name, cfg)
    print(f"  {name:28} {sum(p.numel() for p in enc.parameters()):5d} params, "
          f"out {tuple(enc(x).shape)}")

What each adds over plain convolution:

| Variant | Adds | Why |
|---|---|---|
| **Convolutional** | — (baseline conv stack) | learns local temporal patterns |
| **Resnet** | skip connections (`out = f(x) + x`) | gradients flow through the identity path → trains deeper stacks without vanishing |
| **Multi-Filter Convolutional** | several kernel sizes in parallel, concatenated | captures patterns at multiple timescales at once (short + long) |

The **residual** idea (Resnet) is worth internalizing: adding the input back (`f(x) + x`) gives the gradient a direct route around the conv layers, so [02.5](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb)'s backward pass doesn't attenuate through depth — the innovation that made very deep networks trainable. **Multi-filter** is the inception-style idea: don't pick one kernel size, run several and let the network use whichever timescale matters.

## Section 3 — The `neural_data_decoding` implementation

`ConvolutionalEncoder.forward` — the reshape-to-5-D adapter that makes a conv encoder swappable with a GRU:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/conv_encoder.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.strip().startswith("def forward"))
for k in range(i, min(i + 24, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* The flat `(B, W, T*A*C)` input is `reshape`d to 5-D `(B, W, T, A, C)` — the 02.2 finding, in code: nothing produces a channels-first `(B*W, C, T*A)`; the encoder unflattens to the natural per-window structure.
* Kernels run over `(T, A)` with the channel axis handled explicitly — a 2-D convolution over the time×area plane, not a plain Conv1d.
* The output is flattened back so the return is `(B, W, hidden)` — the exact contract from [04.2](04.2_building_a_simple_encoder.ipynb)/[04.3](04.3_rnn_building_blocks.ipynb), which is what lets `cfg.model_name` swap encoders with zero downstream changes.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the length arithmetic

For `nn.Conv1d(4, 8, kernel_size=5, stride=2, padding=2)` on a length-20 input, what's the output length? Derive it (`floor((L + 2·pad − kernel)/stride) + 1`), then check.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
conv = nn.Conv1d(4, 8, kernel_size=5, stride=2, padding=2)
out = conv(torch.randn(1, 4, 20))
L = 20; pad = 2; k = 5; s = 2
print(f"formula: floor(({L} + 2·{pad} − {k})/{s}) + 1 = {(L + 2*pad - k)//s + 1}")
print(f"actual output length: {out.shape[-1]}")

### Exercise 4.2 — interchangeability, proven

Build a `GRU` encoder and a `Convolutional` encoder from configs that share `in_features`/`hidden_sizes`, feed both the same `(2, 5, 80)` input, and show their OUTPUT shapes match (even though param counts differ wildly).

In [ ]:
# Reveal:
shared = {"in_features": 80, "hidden_sizes": [8, 16, 4]}
gru  = build_encoder("GRU", {**shared, "transform": "GRU", "dropout": 0.0,
                             "want_normalization": False, "activation": ""})
conv = build_encoder("Convolutional", {**shared, "samples_per_window": 10, "num_areas": 2,
                     "dropout": 0.0, "want_normalization": "Instance",
                     "activation": "Leaky ReLU", "stride": 2})
xb = torch.randn(2, 5, 80)
print(f"GRU : {tuple(gru(xb).shape)}  ({sum(p.numel() for p in gru.parameters())} params)")
print(f"Conv: {tuple(conv(xb).shape)}  ({sum(p.numel() for p in conv.parameters())} params)")
print("last axis (hidden) matches ⇒ the composite treats them identically")

## Section 5 — Common errors

### `RuntimeError` about channel dimensions in a conv

The channels-second trap ([02.2 §5](../02_numpy_and_pytorch_basics/02.2_array_axis_conventions.ipynb)) — a `(batch, length, channels)` tensor into `Conv1d`. Conv layers want `(batch, channels, length)`.

### `ValueError: in_features must be divisible by samples_per_window * num_areas`

The conv encoder needs `in_features = T·A·C` to recover the channels-per-area count `C`. A mismatched `samples_per_window`/`num_areas` breaks the reshape — they must factor the flat feature size.

### Conv model trains far slower / OOMs vs the GRU

Conv encoders are memory- and compute-heavier — which is why the MATLAB sweep pairs them with gradient accumulation ([02.5 §2.3](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb), Critical Note #18). Lower the micro-batch, not the batch.

### Output length not what you expected

The conv arithmetic (Exercise 4.1) — stride and padding change the length. `padding='same'`-style behavior needs `padding = kernel_size // 2` with stride 1.

### Swapping GRU → Conv breaks a downstream layer

It shouldn't, if `hidden_sizes` match — the output contract is identical (§2.2). If it breaks, a downstream module hard-coded a shape it shouldn't have; the composite is written to be encoder-agnostic.

## Section 6 — Further reading

- [Conv1d docs](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html) — shapes, stride, padding, dilation.
- [ResNet paper](https://arxiv.org/abs/1512.03385) — the residual-connection idea in §2.3.
- [`src/neural_data_decoding/models/conv_encoder.py`](../../src/neural_data_decoding/models/conv_encoder.py) — the reshape adapter and the three variants.

Next up: **[04.5 the bottleneck](04.5_the_bottleneck.ipynb)** — flatten + FC, and why this exact structure sits between encoder and latent.